In [1]:
# Import the necessary Python modules.

# Data Management/Investigation
import pandas as pd
from pandas.api.types import CategoricalDtype # Ordering categories
import requests # For downloading the website
from bs4 import BeautifulSoup # For parsing the website
import numpy as np
import missingno as miss

# Plotting libraries
from plotnine import *
import matplotlib.pyplot as plt

# For pre-processing data 
from sklearn import preprocessing as pp 
from sklearn.compose import ColumnTransformer 

# For splits and CV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold # Cross validation 
from sklearn.model_selection import cross_validate # Cross validation 
from sklearn.model_selection import GridSearchCV # Cross validation + param. tuning.

# Machine learning methods 
from sklearn.naive_bayes import GaussianNB as NB
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.tree import DecisionTreeClassifier as DT
from sklearn.tree import DecisionTreeRegressor as DT_reg
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn import tree # For plotting the decision tree rules

# For evaluating our model's performance
import sklearn.metrics as m

# Pipeline to combine modeling elements
from sklearn.pipeline import Pipeline

# For model interpretation
from sklearn.inspection import (
    permutation_importance,
    partial_dependence, 
    PartialDependenceDisplay, 
    plot_partial_dependence
)

# Misc
# import warnings
# warnings.filterwarnings("ignore")

***

United States Census Bureau’s “Explore Census Data”<br/>
Identify income in the past twelve months by ZIP Code

In [2]:
# Load the Census Data collected from the U.S. Census Bureau.
census_data = pd.read_csv("Census_Data/ACSST5Y2019.S1901_data_with_overlays_2021-11-04T111610.csv", 
                          low_memory = False)

In [3]:
# Preview the head of the data to ensure it was loaded properly.
census_data.head()

,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001M,S1901_C01_002E,S1901_C01_002M,S1901_C01_003E,S1901_C01_003M,S1901_C01_004E,S1901_C01_004M,...,S1901_C04_012E,S1901_C04_012M,S1901_C04_013E,S1901_C04_013M,S1901_C04_014E,S1901_C04_014M,S1901_C04_015E,S1901_C04_015M,S1901_C04_016E,S1901_C04_016M
0,id,Geographic Area Name,Estimate!!Households!!Total,Margin of Error!!Households!!Total,"Estimate!!Households!!Total!!Less than $10,000",Margin of Error!!Households!!Total!!Less than ...,"Estimate!!Households!!Total!!$10,000 to $14,999","Margin of Error!!Households!!Total!!$10,000 to...","Estimate!!Households!!Total!!$15,000 to $24,999","Margin of Error!!Households!!Total!!$15,000 to...",...,Estimate!!Nonfamily households!!Median income ...,Margin of Error!!Nonfamily households!!Median ...,Estimate!!Nonfamily households!!Mean income (d...,Margin of Error!!Nonfamily households!!Mean in...,Estimate!!Nonfamily households!!PERCENT ALLOCA...,Margin of Error!!Nonfamily households!!PERCENT...,Estimate!!Nonfamily households!!PERCENT ALLOCA...,Margin of Error!!Nonfamily households!!PERCENT...,Estimate!!Nonfamily households!!PERCENT ALLOCA...,Margin of Error!!Nonfamily households!!PERCENT...
1,8600000US00601,ZCTA5 00601,5509,189,37.2,3.8,14.7,2.8,17.9,3.0,...,9419,2104,12857,2026,(X),(X),(X),(X),20.4,(X)
2,8600000US00602,ZCTA5 00602,12740,443,30.0,2.9,15.3,2.4,19.0,2.3,...,9963,1449,14634,2640,(X),(X),(X),(X),13.4,(X)
3,8600000US00603,ZCTA5 00603,19228,503,34.8,2.0,13.1,1.6,16.3,1.9,...,10520,1005,18759,2769,(X),(X),(X),(X),30.4,(X)
4,8600000US00606,ZCTA5 00606,1946,176,41.4,5.7,13.0,4.1,18.0,5.1,...,10272,2285,11572,2117,(X),(X),(X),(X),17.0,(X)


In [4]:
# View the data types of the Census Data.
census_data.dtypes

GEO_ID            object
NAME              object
S1901_C01_001E    object
S1901_C01_001M    object
S1901_C01_002E    object
                   ...  
S1901_C04_014M    object
S1901_C04_015E    object
S1901_C04_015M    object
S1901_C04_016E    object
S1901_C04_016M    object
Length: 130, dtype: object

***

American Hospital Directory (AHD) - List of Hospital Websites and Additional Information

In [21]:
# Prepare list to hold the UN information.
scraped_data = []

# AHD website - CA search results for all hospitals.
url = "https://www.ahd.com/list_cms.php?mstate%5B%5D=CA&listing=1&viewmap=0"

# Download the webpage.
page = requests.get(url)

# If a connection was successfully reached.
if page.status_code == 200:
    # Parse the webpage.
    soup = BeautifulSoup(page.content, "html.parser")
    
    # Identify all table rows on the webpage.
    # Ignore the first seven rows.
    table_rows = soup.find_all("tr")[7:485]

    # Iterate through each table row.
    for table_row in table_rows:
        # Identify all cells within the table row.
        table_cell = table_row.find_all("td")
        
        # Hospital Name.
        name = str(table_cell[0].text)
        
        # Hospital Beds.
        beds = int(table_cell[1].text)
        
        # Hospital City.
        city = str(table_cell[2].text)

        # Pull the items in the cell.
        html_soup = BeautifulSoup(str(table_cell))

        # Iterate through the cell and find all a-tags with href.
        for href_url in html_soup.find_all("a", href = True):
            # Pull the href from the table for only the first entry.
            website = "https://www.ahd.com" + href_url["href"]
            break
        
        # Add row to dataframe.
        scraped_data.append([name, beds, city, website])

# Convert the holding list to Pandas DataFrame.
dat = pd.DataFrame(scraped_data, columns = ["name", "beds", "city", "website"])

# Save the hospital information to a CSV.
dat.to_csv("Hospital_Data/AHD_list.csv")

/free_profile/054074/_Adventist_Health_Saint_Helena_Center_for_Behavioral_Health/Vallejo/California/
/free_profile/050236/_Adventist_Health_Simi_Valley_/Simi_Valley/California/
/free_profile/I41022/_California_Department_of_State_Hospitals_-_Coalinga/Coalinga/California/
/free_profile/054122/_California_Department_of_State_Hospitals_-_Napa/Napa/California/
/free_profile/I41899/_California_Department_of_State_Hospitals_Salinas_Valley_Hospital/Soledad/California/
/free_profile/050139/_Downey_Medical_Center/Downey/California/
/free_profile/050360/_MarinHealth_Medical_Center/Greenbrae/California/
/free_profile/050290/_Providence_Saint_John%27s_Health_Center/Santa_Monica/California/
/free_profile/050677/_Woodland_Hills_Medical_Center/Woodland_Hills/California/
/free_profile/05015F/60th_Medical_Group_-_David_Grant_USAF_Medical_Center/Travis_Air_Force_Base/California/
/free_profile/050133/Adventist_Health_and_Rideout/Marysville/California/
/free_profile/050455/Adventist_Health_Bakersfield_/Ba

/free_profile/050534/John_F_Kennedy_Memorial_Hospital/Indio/California/
/free_profile/I41656/John_George_Psychiatric_Hospital_/San_Leandro/California/
/free_profile/054131/John_Muir_Behavioral_Health_Center/Concord/California/
/free_profile/054147/Joyce_Eisenberg-Keefer_Medical_Center/Reseda/California/
/free_profile/054150/Kaiser_Permanente_Behavioral_Health_Center_in_Santa_Clara/Santa_Clara/California/
/free_profile/050512/Kaiser_Permanente_Fremont_Medical_Center/Fremont/California/
/free_profile/050772/Kaiser_Permanente_Roseville_Medical_Center/Roseville/California/
/free_profile/050425/Kaiser_Permanente_Sacramento_Medical_Center/Sacramento/California/
/free_profile/I42191/Kaiser_Permanente_San_Diego_Medical_Center/San_Diego/California/
/free_profile/050076/Kaiser_Permanente_San_Francisco_Medical_Center/San_Francisco/California/
/free_profile/050070/Kaiser_Permanente_South_San_Francisco_Medical_Center_/South_San_Francisco/California/
/free_profile/050515/Kaiser_Permanente_Zion_Medic

/free_profile/050093/Saint_Agnes_Medical_Center/Fresno/California/
/free_profile/050129/Saint_Bernardine_Medical_Center/San_Bernardino/California/
/free_profile/050042/Saint_Elizabeth_Community_Hospital/Red_Bluff/California/
/free_profile/050104/Saint_Francis_Medical_Center/Lynwood/California/
/free_profile/050152/Saint_Francis_Memorial_Hospital/San_Francisco/California/
/free_profile/050616/Saint_John%27s_Pleasant_Valley_Hospital/Camarillo/California/
/free_profile/050082/Saint_John%27s_Regional_Medical_Center/Oxnard/California/
/free_profile/I42090/Saint_Joseph_Hospital_Acute_Rehabilitation_Unit/Eureka/California/
/free_profile/050006/Saint_Joseph_Hospital_Eureka/Eureka/California/
/free_profile/050069/Saint_Joseph_Hospital_Orange/Orange/California/
/free_profile/054123/Saint_Joseph%27s_Behavioral_Health_Center/Stockton/California/
/free_profile/050084/Saint_Joseph%27s_Medical_Center/Stockton/California/
/free_profile/050168/Saint_Jude_Medical_Center/Fullerton/California/
/free_profi